# cvprofiles — OVB sensitivity on a surviving measure (sensemakr-style)

cvprofiles answers: *which operationalizations are admissible, and what range of the target
functional follows?* It does **not** answer a different, downstream question: *for one fixed
measure's regression coefficient, how much unobserved confounding would overturn it?*

That is the omitted-variable-bias (OVB) sensitivity question of Cinelli and Hazlett (2020),
popularized in the `sensemakr` R package. This notebook runs a cvprofiles profile, takes a
survivor's OLS coefficient, and applies a **hand-rolled** Cinelli–Hazlett bound — bias
formula plus robustness value — using numpy only, so every step is auditable.



In [ ]:
from __future__ import annotations
import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

import cvprofiles
from cvprofiles.pipeline import run_profile

print("cvprofiles", cvprofiles.__version__)



## 1. A DGP with an unobserved confounder

A candidate measure $m$ predicts outcome $y$, but an **unobserved** confounder $u$ affects
both. A naive regression of $y$ on $m$ (with observed control $w$) is therefore biased; the
bias is exactly what the CH framework quantifies.

$$y = \tau m + \gamma_w w + \gamma_u u + \varepsilon, \qquad m = \alpha_w w + \alpha_u u + \eta.$$

We also add a *slop* measure that fails the network, so the profile has a rejection story.



In [ ]:
rng = np.random.default_rng(20260806)
n = 500

u = rng.normal(size=n)            # unobserved confounder
w = rng.normal(size=n)            # observed control

# measure driven by observed control + confounder
m_good = 0.7 * w + 0.8 * u + rng.normal(size=n)
m_slop = 0.9 * rng.normal(size=n)  # unrelated noise -> should fail the network

tau, gw, gu = 0.6, 0.4, 0.7
y = tau * m_good + gw * w + gu * u + rng.normal(size=n)

# external auxiliary for the network: co-moves with the construct-bearing measure
v_aux = 0.6 * m_good + rng.normal(size=n)

print("corr(m_good, y):", round(float(np.corrcoef(m_good, y)[0, 1]), 3))



In [ ]:
def ols_coef(X, y):
    """Numpy OLS coefficients with intercept; returns beta, residual sd."""
    A = np.column_stack([np.ones(len(y)), X])
    beta, *_ = np.linalg.lstsq(A, y, rcond=None)
    resid = y - A @ beta
    sd = float(np.sqrt(np.mean(resid**2) * len(y) / max(len(y) - A.shape[1], 1)))
    return beta, sd

def partial_r2(y, D, X):
    """Partial R2 of D with y given X (and intercept)."""
    Xa = np.column_stack([np.ones(len(y)), X])
    _, sd_restricted = ols_coef(Xa[:, 1:], y)
    A = np.column_stack([Xa, D])
    _, sd_full = ols_coef(A[:, 1:], y)
    rss_restricted = sd_restricted**2 * (len(y) - Xa.shape[1])
    rss_full = sd_full**2 * (len(y) - A.shape[1])
    return float((rss_restricted - rss_full) / rss_restricted)

def ch_bias(r2_y_z, r2_d_z, sd_y, sd_d):
    """Cinelli-Hazlett |bias| from partial R2s and residual sds."""
    return float(np.sqrt(r2_y_z * r2_d_z / (1 - r2_d_z)) * sd_y / sd_d)

def rv_q(tstat, df, q=1.0):
    """Robustness value for a q-fraction reduction of the estimate (CH 2020)."""
    fq = q * abs(tstat) / np.sqrt(df)
    return float(0.5 * (np.sqrt(fq**4 + 4 * fq**2) - fq**2))



## 2. Run the cvprofiles profile

The network requires a valid measure to correlate at least $\theta=0.3$ with the external
auxiliary. The target is the standardized OLS coefficient of $y$ on each measure, controlling
for $w$.



In [ ]:
work = Path(tempfile.mkdtemp(prefix="cvp_sm_"))

scores = pd.DataFrame({
    "unit_id": [f"u{i:04d}" for i in range(n)],
    "m_good": m_good,
    "m_slop": m_slop,
    "v_aux": v_aux,
    "w": w,
    "y": y,
})
scores.to_csv(work / "scores.csv", index=False)

roles = {
    "unit_id": "unit_id",
    "measures": ["m_good", "m_slop"],
    "aux": ["v_aux", "w"],
    "outcome": "y",
    "diagnostic": [],
}
(work / "roles.json").write_text(json.dumps(roles))

network = {
    "schema_version": "1",
    "name": "ovb_oracle",
    "delta": 0.0,
    "restrictions": [
        {"id": "r_corr_min_aux", "type": "corr_min", "theta": 0.3,
         "params": {"variable": "v_aux"}},
    ],
}
(work / "network.yaml").write_text(yaml.safe_dump(network, sort_keys=False))

beta = {
    "schema_version": "1",
    "type": "ols_coef",
    "outcome": "y",
    "params": {"controls": ["w"]},
}
(work / "beta.yaml").write_text(yaml.safe_dump(beta, sort_keys=False))

result = run_profile(
    scores=work / "scores.csv",
    roles=work / "roles.json",
    network=work / "network.yaml",
    beta=work / "beta.yaml",
    out_dir=work / "run",
    seed=0,
    title="OVB sensitivity on survivors",
)

print("M*    :", result.identify.admissible)
print("beta  :", {m: round(float(v), 3) for m, v in result.identify.beta_values.items()})
print("[L,U] :", result.identify.range_L, result.identify.range_U)



## 3. OVB sensitivity on the survivor

The profile admits `m_good`; its **standardized** OLS coefficient (controlling for `w`) is the
estimate we care about. We now ask the CH question: *how strong would an unobserved confounder
$u^*$ need to be — in partial-$R^2$ units — to move this estimate by $q$ of its value?*

First, reproduce the *known* confounder case: because we simulated $u$, we can compute its
true partial $R^2$s and confirm the OVB identity recovers the true coefficient. Then report
the robustness value for a hypothetical confounder. Everything below uses z-scored variables,
so the numbers line up with the profile's standardized $\beta$.



In [ ]:
def z(v):
    return (v - v.mean()) / v.std()

mz, wz, uz, yz = z(m_good), z(w), z(u), z(y)

# survivor regression: y ~ m_good + w  (restricted; u omitted), standardized
m_hat, sd_y_mw = ols_coef(np.column_stack([mz, wz]), yz)
beta_res = float(m_hat[1])

# treatment regression: m_good ~ w (standardized)
m_hat_d, sd_d_w = ols_coef(wz[:, None], mz)

# true partial R2s of the omitted u
r2_y_u = partial_r2(yz, uz, np.column_stack([mz, wz]))
r2_d_u = partial_r2(mz, uz, wz)

# EXACT OVB identity: bias = gamma_hat * delta_hat (impact x imbalance).
# gamma: coefficient on u in y ~ m_good + w + u (impact on outcome).
# delta: coefficient on m_good in u ~ m_good + w (imbalance: how the
#        omitted confounder predicts the treatment, given controls).
full_y, _ = ols_coef(np.column_stack([mz, wz, uz]), yz)
gamma_hat = float(full_y[3])
aux_u, _ = ols_coef(np.column_stack([mz, wz]), uz)
delta_hat = float(aux_u[1])
bias_ovb = gamma_hat * delta_hat
adj_exact = beta_res - bias_ovb
tau_true = float(full_y[1])

# CH partial-R2 reparameterization (same bias, expressed in R2 units)
bias_ch = ch_bias(r2_y_u, r2_d_u, sd_y_mw, sd_d_w)

print("profile beta (std) :", round(float(result.identify.beta_values["m_good"]), 3))
print("restricted beta    :", round(beta_res, 3))
print("bias (gamma*delta) :", round(bias_ovb, 3))
print("CH |bias| (R2)     :", round(bias_ch, 3))
print("adjusted beta      :", round(adj_exact, 3))
print("true tau (full)    :", round(tau_true, 3))
print("exact recovery     :", round(abs(adj_exact - tau_true), 10) < 1e-8)
print("CH approx matches  :", round(abs(bias_ch - abs(bias_ovb)), 3) < 0.01)



In [ ]:
# Robustness value for a hypothetical confounder: q = 1 means "to zero".
# t-stat of the restricted treatment coefficient
se_beta = None
A = np.column_stack([np.ones(n), m_good, w])
beta_all, *_ = np.linalg.lstsq(A, y, rcond=None)
resid = y - A @ beta_all
sigma2 = float(resid @ resid / (n - A.shape[1]))
cov = np.linalg.inv(A.T @ A) * sigma2
se_beta = float(np.sqrt(cov[1, 1]))
tstat = beta_res / se_beta
df = n - A.shape[1]

rv1 = rv_q(tstat, df, q=1.0)
print("survivor t-stat:", round(tstat, 2), "| df:", df)
print("RV (q=1)       :", round(rv1, 3),
      "=> a confounder explaining", round(rv1 * 100, 1),
      "% of residual variance of BOTH treatment and outcome would zero the estimate")



## 4. What OVB adds — and what it does not

- **It adds:** a quantitative, auditable statement about a *fixed* survivor's coefficient.
  The exact OVB identity decomposes the bias into the confounder's impact ($\gamma$) times its
  imbalance ($\delta$); the Cinelli–Hazlett reparameterization expresses the same bias in
  partial-$R^2$ units, and the robustness value summarizes how strong omitted confounding must
  be to overturn the estimate — exactly the `sensemakr` quantities.
- **It does not:** decide which measures are admissible. That is the nomological network's
  job, and it is done *before* this analysis. The slop measure is already excluded by the
  profile; we never run OVB sensitivity on it as if it were a live estimate.
- **Boundary:** this is sensitivity analysis for a regression coefficient, not causal
  identification of a structural effect, and not a substitute for the measurement layer.



In [ ]:
# self-checking assertions
assert set(result.identify.admissible) == {"m_good"}
assert "m_slop" in result.identify.rejected
assert abs(adj_exact - tau_true) < 1e-8          # exact OVB identity recovers full-model tau
assert abs(bias_ch - abs(bias_ovb)) < 0.01       # CH partial-R2 reparameterization matches
assert 0 < rv1 < 1                               # RV is a fraction
assert result.identify.range_L is not None
print("all sensemakr tutorial assertions passed")



## What to look at next

- The hand-rolled functions here are the `sensemakr` core in ~30 lines: partial $R^2$, the CH
  bias formula, and the robustness value. For contour plots and benchmarking against observed
  covariates, the R/Stata package or a Python port adds convenience, not new math.
- cvprofiles disciplines *which* measure; OVB sensitivity disciplines *one fixed* estimate.
  They are complementary, not competing.

